# Image Evals Are Simpler Than You Think

*Plain text is spoken. Code cells are on screen. Italic lines are screen cues. Headings are not spoken.*

In this video, we grade images that models made, and each grade is one more column next to its image, in the tables that already hold the images.

The method comes from OpenAI's [image evals cookbook](https://developers.openai.com/cookbook/examples/multimodal/image_evals), which grades an image in three steps. To run it, OpenAI built what they call a vision eval harness.

*On screen: the cookbook's three steps: hard gates first, then scores from 0 to 5, then tags for how an image failed.*

*On screen: the flyer from our run, with its top cut off.*

Image models fail quietly. A bad image looks finished until someone reads it.

The usual eval is a script next to the app. Someone keeps it running instead of the product, and new images go ungraded until it runs again.

*On screen: the repo.*

We'll use Pixeltable, a backend for AI apps on images and video, where a column can call a model. There are four recipes, and the code is in the repo linked below.

## Install the version this repo pins

Start Jupyter with `uv run jupyter lab` in this folder, with your OpenAI and Gemini keys set in your shell. `uv sync` installs Pixeltable 0.7.12, the version in this repo's lock file.

In [ ]:
!uv sync

## The judge is one more column next to the image

The tables are declared in `schema.py`. In `generations`, you fill in every column above `image`. Pixeltable computes `image` by calling OpenAI's image model.

In [ ]:
!sed -n '42,51p' schema.py

The grade lives in `gen_evals`, which holds only the `generations` rows where `eval_ready` is true. Its `judge_raw` column sends each image to GPT-5.2, and the columns under it read that one answer.

In [ ]:
!sed -n '77,103p' schema.py

`pxt schema diff` shows what will change, and `pxt schema update` creates the tables.

In [ ]:
!pxt schema diff schema.py img_eval

In [ ]:
!pxt schema update schema.py img_eval -f

## Recipe 1: Mark images ready, and the judge grades the ones already in the table

The seed script inserts four rows, so the image models make two images and two edits. That's four paid image calls, about a minute in our run. It leaves `eval_ready` false, so nothing gets graded yet.

In [ ]:
!python scripts/seed_demo.py

In [ ]:
import udfs  # register the judge functions before opening the tables
import pixeltable as pxt

generations = pxt.get_table('img_eval/generations')
edits = pxt.get_table('img_eval/edits')
gen_evals = pxt.get_table('img_eval/gen_evals')
edit_evals = pxt.get_table('img_eval/edit_evals')

generations.select(generations.image, generations.case_id, generations.workflow).collect()

Here are the checkout screen and the flyer, with no grades yet.

### The judge's prompt is built from the row

Before we pay for a grade, let's look at what the judge gets. `judge_user_text` fills in the row's `prompt` and `criteria`. The row's `workflow` picks the rubric, which gives the instructions and the JSON schema.

In [ ]:
!sed -n '92,112p' udfs.py

Here's that prompt filled in for the checkout screen. We run the same functions as a query, so no model gets called.

In [ ]:
row = generations.where(generations.case_id == 'ui_checkout_mockup').select(
    instructions=udfs.judge_instructions(generations.workflow),
    user_text=udfs.judge_user_text(generations.prompt, generations.criteria),
    text_format=udfs.judge_text_format(generations.workflow),
).collect()[0]

print(row['instructions'][:330], '...\n')
print(row['user_text'], '\n')
row['text_format']

The instructions come from the checkout rubric, and the user text is this row's prompt and criteria. The schema sets the fields the judge returns, and the image goes in next to the text.

Marking the images ready is one `update`. The judge grades both images already in the table, and no image gets made again.

In [ ]:
generations.update({'eval_ready': True}, where=generations.batch_id == 'demo_v1')

gen_evals.select(
    gen_evals.image, gen_evals.case_id, gen_evals.verdict,
    gen_evals.scores, gen_evals.tags,
).collect()

The checkout screen passes, with fives for layout and for buttons that look tappable. The flyer fails. Its top is cut off, and the judge flags the shop name, Sunrise Coffee, as missing. Your images will differ, so read your own reasons.

## Recipe 2: The flyer's exact words get a plain check

A judge can miss a word, so the flyer gets a second check. A model reads each line of text off the flyer, and plain Python compares those lines with the required list.

In [ ]:
gen_evals.where(gen_evals.workflow == 'marketing_flyer').select(
    gen_evals.extracted_text, gen_evals.text_check
).collect()

The plain check finds all five required lines, plus an extra word, COFFEE. The judge failed the flyer for the shop name, which isn't on the list, and the verdict fails if either check does.

## Recipe 3: An edit is graded against the images it started from

An edit can only be graded next to what it started from. `edit_judge_content` puts the reference images in the judge's message, ahead of the edited result.

*On screen: `schema.py` lines 61–74 (the `edits` table and its Gemini call) and 109–116 (the columns that fill `edit_judge_content`).*

In [ ]:
!sed -n '68,88p' udfs.py

Gemini made two edits: the logo, from one reference image, and the try-on, from two. Marking them ready grades both.

In [ ]:
edits.update({'eval_ready': True}, where=edits.batch_id == 'demo_v1')

edit_evals.select(
    edit_evals.image, edit_evals.case_id, edit_evals.verdict,
    edit_evals.scores, edit_evals.tags,
).collect()

The try-on passes. The logo fails on all three scores. The edit dropped the ampersand and turned the gray background white.

A new edit gets made and graded in one insert. This instruction changes the line under the name, which the seed edit left alone.

In [ ]:
logo_prompt = (
    'Edit the logo by changing the text LOCAL BAKERY to FRESH DAILY.\n'
    'Do not change any other text, colors, shapes, or layout.'
)

edits.insert(
    case_id='logo_tagline_edit',
    batch_id='run_2',
    workflow='logo_edit',
    gen_prompt=logo_prompt,
    prompt=logo_prompt,
    criteria=(
        'The requested edit is applied exactly.\n'
        'All non-target content remains unchanged.\n'
        'Character style, color, and geometry remain consistent with the original.'
    ),
    ref_image='use_cases/logo_edit/assets/logo_input.png',
    eval_ready=True,
)

edit_evals.where(edit_evals.case_id == 'logo_tagline_edit').select(
    edit_evals.ref_image, edit_evals.image, edit_evals.verdict, edit_evals.scores, edit_evals.tags
).collect()

FRESH DAILY passes, with a 4 for leaving the rest alone. The judge saw a slight change in the background and let it through.

## Recipe 4: Tighten a rubric, and only the verdicts rerun

The judge gives the scores, and the rubric's Python turns them into a verdict. Right now the logo rubric passes an edit when every score is 4 or higher.

In [ ]:
!sed -n '140,155p' use_cases/logo_edit/rubric.py

We'll tighten it, so every score has to be a 5. That catches drift like the background change in FRESH DAILY.

In [ ]:
from pathlib import Path

rubric = Path('use_cases/logo_edit/rubric.py')
rubric.write_text(rubric.read_text().replace('or 0) < 4', 'or 0) < 5'))
!git diff use_cases/logo_edit/rubric.py

Python loaded the rubric when the kernel started, so restart the kernel to pick up the edit.

*Kernel > Restart Kernel.*

### Restart the kernel, and every image and grade is still there

The notebook has no variables now. Open the tables again, and every image and grade is still there, because they're stored in the tables, not in the notebook.

In [ ]:
import udfs  # register the judge functions before opening the tables
import pixeltable as pxt

edit_evals = pxt.get_table('img_eval/edit_evals')

edit_evals.select(edit_evals.image, edit_evals.case_id, edit_evals.verdict, edit_evals.tags).collect()

The threshold is plain Python, so only `verdict` and `tags` need to rerun. We note the table's version first, so we can read the old grades afterward.

In [ ]:
before = edit_evals.get_metadata()['version']

status = edit_evals.recompute_columns('verdict', 'tags')
status.updated_cols

Only `verdict` and `tags` were updated. Every update makes a new version of the table, and `pxt history` lists them.

In [ ]:
!pxt history img_eval/edit_evals -n 3

An older version opens by its number. Here are the old verdicts next to the new ones, and the judge's response IDs show it's the same answer.

In [ ]:
old = pxt.get_table(f'img_eval/edit_evals:{before}')
old_rows = old.select(
    old.case_id, old_verdict=old.verdict, old_response=old.judge_raw['id']
).collect().to_pandas()
new_rows = edit_evals.select(
    edit_evals.case_id, edit_evals.verdict, edit_evals.tags, response=edit_evals.judge_raw['id']
).collect().to_pandas()

both = old_rows.merge(new_rows, on='case_id')
both['same_judge_answer'] = both.old_response == both.response
both[['case_id', 'old_verdict', 'verdict', 'tags', 'same_judge_answer']]

FRESH DAILY fails now, tagged `edit_spill` and `style_drift`. The judge's answers are the same ones as before, so the re-grade made no judge calls and no new images.

A change to the judge's prompt or JSON schema is different. That reruns `judge_raw`, a judge call on every row, and everything under it.

In [ ]:
!pxt recompute img_eval/edit_evals judge_raw -n

Three rows means three judge calls. The images still don't get made again.

## Put the rubric back and delete the sample rows

To finish, put the rubric back the way it was and delete the rows we added.

In [ ]:
!git checkout -- use_cases/logo_edit/rubric.py

generations = pxt.get_table('img_eval/generations')
edits = pxt.get_table('img_eval/edits')
generations.delete(where=generations.batch_id.isin(['demo_v1', 'run_2']))
edits.delete(where=edits.batch_id.isin(['demo_v1', 'run_2']))

## Every grade sits next to its image

We graded two images and three edits, and every grade sits in a column next to its image, in the tables that made the images.

| Recipe | What gets graded | In our run |
|---|---|---|
| 1. Mark images ready | the checkout screen and the flyer, already in the table | checkout passes; the flyer fails, its shop name cut off |
| 2. Exact words | the flyer's text, line by line | all five required lines found, plus one extra word |
| 3. Edits | the logo and the try-on, then a new logo edit on insert | the try-on and FRESH DAILY pass; the first logo edit fails |
| 4. A tighter rubric | the same edits, verdicts only | FRESH DAILY fails now, with no judge calls |

Each job folder in the repo has its own `rubric.py`, so start from the one closest to your images. A star on the Pixeltable repo helps other developers find the project.